# EDA

**Autor:** duac1011 · **Datum:** 2026-09-28

**Ziel:** Die Trainingsdaten werden explorativ untersucht, um Datenqualität, Verteilungen und erste Modellierungsentscheidungen abzuleiten.

## 1. Umgebung einrichten

Autoreload aktivieren und die benötigten Bibliotheken sowie Projektmodule importieren.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from awp2.config import FIGURES_DIR, SEED
from awp2.data import band_columns, load_train, wavelengths
from awp2.plots import plot_spectra, save_doc_figure

print(f"Analyseabbildungen: {FIGURES_DIR}")

## 2. Trainingsdaten laden

Die validierten Trainingsdaten laden und die Spektralbandspalten in Wellenlängenreihenfolge bestimmen.

In [ ]:
train = load_train()
bands = band_columns(train)
train.shape

## 3. Datenstruktur untersuchen

Form, Spalten, Datentypen und deskriptive Statistiken geben einen ersten Überblick über Metadaten, Zielvariablen und Spektralbänder.

In [ ]:
print(f"Zeilen: {train.shape[0]:,}")
print(f"Spalten: {train.shape[1]:,}")
print(f"Spektralbänder: {len(bands)}")

display(train.head())
display(train.dtypes.value_counts().rename("Anzahl Spalten").to_frame())
display(train[["AEZ", "Month", *bands]].describe().T)

## 4. Fehlende Werte analysieren

Fehlende Werte je Spalte quantifizieren. Vollständig leere Bänder und vereinzelte Lücken werden getrennt sichtbar gemacht.

In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing_share = (missing / len(train) * 100).rename("Anteil [%]")
missing_summary = pd.concat([missing.rename("Anzahl"), missing_share], axis=1)

display(missing_summary)

fig, ax = plt.subplots(figsize=(12, 5))
missing.plot.bar(ax=ax)
ax.set(title="Fehlende Werte je betroffener Spalte", xlabel="Spalte", ylabel="Anzahl")
ax.tick_params(axis="x", labelsize=7)
fig.tight_layout()

## 5. Zielvariablen auswerten

`Crop` und `Stage` sind kategorial. Statt numerischer Lage- und Streuungsmaße werden absolute und relative Häufigkeiten, Klassenungleichgewichte sowie vorhandene Crop-Stage-Kombinationen untersucht.

In [ ]:
for target in ("Crop", "Stage"):
    counts = train[target].value_counts()
    shares = train[target].value_counts(normalize=True).mul(100)
    display(pd.concat([counts.rename("Anzahl"), shares.rename("Anteil [%]")], axis=1))

crop_stage_counts = pd.crosstab(train["Crop"], train["Stage"])
display(crop_stage_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train["Crop"].value_counts().plot.bar(ax=axes[0], title="Verteilung der Kulturen")
train["Stage"].value_counts().plot.bar(ax=axes[1], title="Verteilung der Stadien")
for ax in axes:
    ax.set(xlabel="", ylabel="Anzahl")
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

## 6. Spektraldaten visualisieren

Zunächst werden zufällig ausgewählte Einzelspektren je Kultur dargestellt. Danach zeigt der Projekt-Helper die mittleren Spektren mit einer Standardabweichung; diese Abbildung wird für die Dokumentation gespeichert.

In [ ]:
sampled = train.groupby("Crop", observed=True, group_keys=False).sample(
    n=3, random_state=SEED
)
x = np.asarray(wavelengths(train))

fig, ax = plt.subplots(figsize=(12, 5))
for _, row in sampled.iterrows():
    ax.plot(x, row[bands].to_numpy(dtype=float), alpha=0.45, linewidth=0.8)
ax.set(
    title="Ausgewählte Spektren je Kultur",
    xlabel="Wellenlänge [nm]",
    ylabel="Reflektanz [%]",
)
ax.grid(alpha=0.3)
fig.tight_layout()

ax = plot_spectra(
    train,
    by="Crop",
    std=True,
    title="Mittlere Spektren nach Kultur (± 1 Std)",
)
save_doc_figure(ax.figure, "spektren_nach_kultur")